# Assignment 3: The "Multimodal Sentiment Engine" Challenge

**Total Marks:** 20 | **Deadline:** 7:00 PM, 22nd March, 2026 |
**Submission:** A zip file of the folder containing this notebook, and the csv/image files you will create.


---

## Setup

Run the cell below **once** to install all required packages and download NLTK data.

In [ ]:
!pip install gdown

!gdown --folder https://drive.google.com/drive/folders/1uV0rVUOxg9u4QDedf1wP1Wn61dpRLqfY

Retrieving folder contents
Processing file 1tRzGtmrsTkwP8uqd53PWCxn_m8asGHEv evaluator.py
Processing file 1Lu7uCBYidK6nuBNroxiVmwtcbefpVWNG gold_standard_100.csv
Processing file 1fJ-YBr2Vs97wk51udUT65ONWbYX86VNB llm_labels_150.csv
Processing file 1VN6MEy8_pcM-QE3Ujs1mPka6Nd8WcBjf requirements.txt
Processing file 1b08aAeLtJ000PA_M5VM4k73G8euOXXh3 weak_labels_200.csv
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1tRzGtmrsTkwP8uqd53PWCxn_m8asGHEv
From (redirected): https://drive.google.com/uc?id=1tRzGtmrsTkwP8uqd53PWCxn_m8asGHEv&confirm=t&uuid=c38deadb-a3d6-4731-93dd-87ed7638d1e2
To: /content/STT_A3/evaluator.py
100% 2.44k/2.44k [00:00<00:00, 6.73MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Lu7uCBYidK6nuBNroxiVmwtcbefpVWNG
To: /content/STT_A3/gold_standard_100.csv
100% 10.6k/10.6k [00:00<00:00, 24.0MB/s]
Downloading...
From: https://drive.google.com/uc?id

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!pip install --upgrade --force-reinstall nltk -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 804.6/804.6 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 2.8 MB/s eta 0:00:00


In [ ]:
import os
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"


In [ ]:
!pip install -r requirements.txt -q

import nltk
nltk.pathsec.ALLOW_PROXIED_FETCH = True # Added this line
for pkg in ["wordnet", "averaged_perceptron_tagger_eng", "punkt_tab", "omw-1.4"]:
    nltk.download(pkg, quiet=True)
print("Setup complete!")

Setup complete!


In [ ]:
from dotenv import load_dotenv
import os

key = input("Enter OpenRouter API key (will be saved only in .env): ")

with open(".env", "w") as f:
    f.write(f"OPENROUTER_API_KEY={key}\n")

load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")
print("API key loaded:", API_KEY is not None)

In [ ]:
import os, re, json, time, random, warnings
from collections import Counter
from itertools import combinations

from dotenv import load_dotenv
load_dotenv()

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

warnings.filterwarnings("ignore")

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
LLM_MODEL = "google/gemini-2.5-flash"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Sentiment-bearing words to preserve during augmentation
PRESERVE_WORDS = {
    "amazing", "terrible", "awful", "excellent", "wonderful", "horrible",
    "great", "bad", "good", "worst", "best", "love", "hate", "boring",
    "fantastic", "brilliant", "pathetic", "outstanding", "dreadful",
    "superb", "mediocre",
}

print("Imports loaded. API key present:", bool(OPENROUTER_API_KEY))

Imports loaded. API key present: True


## Task 1: Data Consolidation & Classical Augmentation (5 Marks)

**Steps:**
1. Load all three CSVs and merge them
2. Train a baseline Logistic Regression on `gold_standard_100.csv` (TF-IDF features)
3. Filter `llm_labels_150.csv` -- keep only reviews where baseline confidence ≥ 0.65 AND agrees with LLM label
4. Deduplicate by review text $\rightarrow$ save `consolidated_base.csv`
5. Identify minority class, apply 2 augmentation methods (Synonym Replacement, Back Translation)
6. Quality filter augmented samples (Jaccard similarity)
7. Save `augmented_classical.csv` and `class_distribution.png`

In [ ]:
gold = pd.read_csv("STT_A3/gold_standard_100.csv")
weak = pd.read_csv("STT_A3/weak_labels_200.csv")
llm  = pd.read_csv("STT_A3/llm_labels_150.csv")
print(f"Gold: {len(gold)}, Weak: {len(weak)}, LLM: {len(llm)}")

def train_baseline_model(train_df, text_col="review", label_col="label"):
    """Returns (vectorizer, classifier) trained on the given dataframe."""
    vec = TfidfVectorizer(max_features=5000, stop_words="english")
    X = vec.fit_transform(train_df[text_col])
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X, train_df[label_col])
    return vec, clf

vec, clf = train_baseline_model(gold)

#  1c. Filter LLM labels by confidence
# TODO: Predict on llm reviews, keep where confidence >= 0.65 AND prediction matches LLM label
X_llm = vec.transform(llm["review"])
probs = clf.predict_proba(X_llm)
preds = clf.predict(X_llm)
max_conf = probs.max(axis=1)

llm_filtered_mask = (max_conf >= 0.65) & (preds == llm["label"].values)
llm_filtered = llm[llm_filtered_mask].copy()

print(f"LLM total: {len(llm)}, filtered kept: {len(llm_filtered)}")

#  1d. Merge & deduplicate
# TODO: Combine gold + weak + filtered_llm, drop_duplicates on "review"
# Save as consolidated_base.csv
consolidated = pd.concat([gold, weak, llm_filtered],axis=0,ignore_index=True)
consolidated = consolidated.drop_duplicates(subset=["review"]).reset_index(drop=True)
print("Consolidated size:", len(consolidated))

consolidated.to_csv("consolidated_base.csv", index=False)

#  1e. Class distribution analysis
# TODO: Count per class, identify minority, plot and save class_distribution.png
counts = Counter(consolidated["label"])
print(counts)
# class_counts = consolidated["label"].value_counts()
# print("Class counts:\n", class_counts)
minority_class = min(counts, key=counts.get)
print("Minority class:", minority_class)

plt.figure(figsize=(8, 6))
# plt.bar(counts.keys(), counts.values())
plt.bar(list(counts.keys()), list(counts.values()))
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.title("Class Distribution")
plt.savefig("class_distribution.png")
plt.show()

Gold: 100, Weak: 220, LLM: 150
LLM total: 150, filtered kept: 27
Consolidated size: 328
Counter({'Negative': 151, 'Neutral': 115, 'Positive': 62})
Minority class: Positive


In [ ]:
from string import punctuation

def nltk_pos_to_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV
    return None

In [ ]:
!pip install deep-translator
import deep_translator
from deep_translator import GoogleTranslator

[nltk_data] Error loading punkt_tab: Security Violation
[nltk_data]     [pathsec.urlopen]: refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be SSRF-safe, opt in via
[nltk_data]     NLTK_ALLOW_PROXIED_URLOPEN=1 or
[nltk_data]     nltk.pathsec.ALLOW_PROXIED_FETCH=True.
[nltk_data] Error loading punkt: Security Violation [pathsec.urlopen]:
[nltk_data]     refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be S

False

In [ ]:

def get_synonym(word, pos_tag=None):
    wn_pos = nltk_pos_to_wordnet_pos(pos_tag) if pos_tag else None
    synsets = wordnet.synsets(word, pos=wn_pos) if wn_pos else wordnet.synsets(word)
    lemmas = []
    for syn in synsets:
        for lemma in syn.lemmas():
            name = lemma.name().replace("_", " ")
            if name.lower() != word.lower():
                lemmas.append(name)
    if not lemmas:
        return None
    return random.choice(lemmas)

def synonym_replacement(text, replace_frac=0.15):
    """Replace ~15% of words with WordNet synonyms, preserving sentiment-bearing words."""
    tokens = word_tokenize(text)
    if not tokens:
        return text

    # POS tagging
    tagged = pos_tag(tokens)

    # candidate indices: exclude punctuation, preserve_words
    candidate_indices = []
    for i, (tok, tag) in enumerate(tagged):
        if tok.lower() in PRESERVE_WORDS:
            continue
        if tok.lower() in punctuation:
            continue
        if not tok.isalpha():
            continue
        candidate_indices.append(i)

    if not candidate_indices:
        return text

    n_to_replace = max(1, int(len(candidate_indices) * replace_frac))
    indices_to_replace = random.sample(candidate_indices, min(n_to_replace, len(candidate_indices)))

    new_tokens = tokens[:]
    for idx in indices_to_replace:
        word = tokens[idx]
        tag = tagged[idx][1]
        syn = get_synonym(word, tag)
        if syn:
            new_tokens[idx] = syn

    return " ".join(new_tokens)

def back_translate(text, src="en", mid="hi"):
    """Translate English → Hindi → English using deep-translator GoogleTranslator."""

    try:
        # forward
        mid_text = GoogleTranslator(source=src, target=mid).translate(text)
        time.sleep(0.2)
        # back
        back_text = GoogleTranslator(source=mid, target=src).translate(mid_text)
        time.sleep(0.2)
        return back_text
    except Exception as e:
        # if translation fails, just return original
        return text

def jaccard_similarity(a, b):
    a_tokens = set(word_tokenize(a.lower()))
    b_tokens = set(word_tokenize(b.lower()))
    if not a_tokens or not b_tokens:
        return 0.0
    intersection = len(a_tokens & b_tokens)
    union = len(a_tokens | b_tokens)
    return intersection / union

def quality_filter(original, augmented, low=0.30, high=0.95):
  """Return True if augmented text passes Jaccard similarity (0.30–0.95)."""
    # TODO: Implement Jaccard similarity check
  sim = jaccard_similarity(original, augmented)
  return (sim >= low) and (sim <= high)

In [ ]:
#  1g. Apply augmentation to minority class
# TODO: For each minority-class sample, generate 2 augmented versions (one per method)
# TODO: Apply quality_filter, keep only passing samples
# TODO: Save augmented_classical.csv

minority_df = consolidated[consolidated["label"] == minority_class].reset_index(drop=True)
print("Minority samples:", len(minority_df))

aug_rows = []

for _, row in minority_df.iterrows():
    orig_text = row["review"]
    label = row["label"]

    # Synonym replacement
    syn_aug = synonym_replacement(orig_text, replace_frac=0.15)
    if syn_aug and syn_aug != orig_text and quality_filter(orig_text, syn_aug):
        aug_rows.append({
            "review": syn_aug,
            "label": label,
            "aug_method": "synonym_replacement"
        })

    # Back translation
    bt_aug = back_translate(orig_text)
    if bt_aug and bt_aug != orig_text and quality_filter(orig_text, bt_aug):
        aug_rows.append({
            "review": bt_aug,
            "label": label,
            "aug_method": "back_translation"
        })

augmented_classical = pd.DataFrame(aug_rows)
print("Augmented rows kept after quality filter:", len(augmented_classical))

augmented_classical.to_csv("augmented_classical.csv", index=False)

Minority samples: 62
Augmented rows kept after quality filter: 58


## Task 2: LLM-Based Synthetic Review Generation (5 Marks)

**Steps:**
1. Design a few-shot prompt with 3-4 gold-standard examples
2. Use OpenRouter API (via `openai` package) to generate 300 synthetic reviews in batches of 20
3. Calculate diversity metrics: Self-BLEU per class
4. Run sentiment consistency check with baseline model $\rightarrow$ flag mismatches
5. Save `llm_generated_300.csv`, `llm_generated_flagged.csv`, `prompt_template.txt`, `diversity_report.txt`

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

#  2a. Design your few-shot prompt
# TODO: Build a prompt string with 3-4 example reviews from gold standard
# Instruct the LLM to output JSON: [{"review": "...", "sentiment": "Positive", "movie": "..."}]
PROMPT_TEMPLATE = """You are a movie review generator. ..."""

# Save prompt to file
with open("prompt_template.txt", "w", encoding="utf-8") as f:
    f.write(PROMPT_TEMPLATE)

#  2b. Generate reviews in batches
# TODO: Loop to generate ~300 reviews in batches of 20
# Target distribution: ~150 Positive, ~100 Negative, ~50 Neutral
# Parse JSON response, handle errors


#  2c. Diversity metrics
# TODO: Calculate Self-BLEU per sentiment class using nltk.translate.bleu_score


#  2d. Sentiment consistency check
# TODO: Use baseline model (vec, clf) to predict sentiment of each generated review
# TODO: Flag mismatches, save llm_generated_flagged.csv


#  2e. Save outputs
# TODO: Save llm_generated_300.csv and diversity_report.txt

In [ ]:
# ====== TASK 2: LLM Synthetic Review Generation ======
from openai import OpenAI
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

# 2a. Build few-shot prompt
examples = gold.sample(4, random_state=RANDOM_SEED)  # 3-4 examples

example_blocks = []
for _, r in examples.iterrows():
    example_blocks.append(
        json.dumps({
            "review": r["review"],
            "sentiment": r["label"],
            "movie": "ExampleMovie"
        }, ensure_ascii=False)
    )

EXAMPLE_JSON = "[\n" + ",\n".join(example_blocks) + "\n]"

PROMPT_TEMPLATE = f"""
You are a movie review generator.

Generate a JSON array of 20 synthetic movie reviews.
Each element must be an object with keys: "review", "sentiment", "movie".

- "review": 1-3 sentences of a realistic movie review.
- "sentiment": one of "Positive", "Negative", or "Neutral".
- "movie": a plausible movie title (can be invented).

Target distribution per batch (approx):
- 10 Positive
- 7 Negative
- 3 Neutral

Here are some examples of the JSON format:
{EXAMPLE_JSON}

Return ONLY valid JSON, no extra text.
"""

with open("prompt_template.txt", "w", encoding="utf-8") as f:
    f.write(PROMPT_TEMPLATE)

def generate_batch():
    """Generate one batch (~20 reviews) from the LLM."""
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": PROMPT_TEMPLATE},
        ],
        temperature=0.9,
        max_tokens=2000,
    )
    content = resp.choices[0].message.content
    # Try to parse JSON
    try:
        data = json.loads(content)
        if isinstance(data, dict):
            data = [data]
        return data
    except Exception:
        # simple recovery: try to extract JSON substring if needed
        try:
            start = content.index("[")
            end = content.rindex("]") + 1
            data = json.loads(content[start:end])
            return data
        except Exception:
            return []

all_gen = []
while len(all_gen) < 300:
    batch = generate_batch()
    print("Batch size:", len(batch))
    for item in batch:
        if "review" in item and "sentiment" in item:
            all_gen.append({
                "review": item["review"],
                "label": item["sentiment"],
                "movie": item.get("movie", "")
            })
    print("Total so far:", len(all_gen))

llm_generated_300 = pd.DataFrame(all_gen[:300])
llm_generated_300.to_csv("llm_generated_300.csv", index=False)
print("Saved llm_generated_300.csv", len(llm_generated_300))


Batch size: 20
Total so far: 20
Batch size: 20
Total so far: 40
Batch size: 20
Total so far: 60
Batch size: 20
Total so far: 80
Batch size: 20
Total so far: 100
Batch size: 20
Total so far: 120
Batch size: 20
Total so far: 140
Batch size: 20
Total so far: 160
Batch size: 20
Total so far: 180
Batch size: 20
Total so far: 200
Batch size: 20
Total so far: 220
Batch size: 20
Total so far: 240
Batch size: 20
Total so far: 260
Batch size: 20
Total so far: 280
Batch size: 20
Total so far: 300
Saved llm_generated_300.csv 300


In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
smooth = SmoothingFunction().method1

def self_bleu_for_class(df, label_col="label", text_col="review"):
    """
    Computes average pairwise BLEU for each sentiment class separately.
    Lower Self-BLEU => more diverse.
    """
    scores = {}
    for sentiment in df[label_col].unique():
        sub = df[df[label_col] == sentiment][text_col].tolist()
        if len(sub) < 2:
            scores[sentiment] = None
            continue
        bleu_vals = []
        # all pairwise combinations i<j
        for i, j in combinations(range(len(sub)), 2):
            ref = sub[i].split()
            hyp = sub[j].split()
            bleu_vals.append(
                sentence_bleu([ref], hyp, smoothing_function=smooth)
            )
        scores[sentiment] = float(np.mean(bleu_vals))
    return scores

# Use it on your generated reviews
self_bleu_scores = self_bleu_for_class(llm_generated_300)
print("Self-BLEU per class:", self_bleu_scores)

# Check target < 0.7 and write report
with open("diversity_report.txt", "w") as f:
    f.write("Self-BLEU per class:\n")
    for sentiment, score in self_bleu_scores.items():
        f.write(f"{sentiment}: {score}\n")
        if score is not None:
            status = "OK" if score < 0.7 else "TOO HIGH"
            print(f"{sentiment}: {score:.3f} ({status})")
            f.write(f"Status: {status} (target < 0.7)\n")
        else:
            print(f"{sentiment}: not enough samples")
            f.write("Status: not enough samples\n")


Self-BLEU per class: {'Positive': 0.0201703027996393, 'Negative': 0.023421897631369513, 'Neutral': 0.026748815239538384}
Positive: 0.020 (OK)
Negative: 0.023 (OK)
Neutral: 0.027 (OK)


In [ ]:
X_gen = vec.transform(llm_generated_300["review"])
pred_gen = clf.predict(X_gen)

flag_mask = pred_gen != llm_generated_300["label"]
llm_generated_flagged = llm_generated_300[flag_mask].copy()
llm_generated_flagged.to_csv("llm_generated_flagged.csv", index=False)
print("Flagged LLM reviews:", len(llm_generated_flagged))

Flagged LLM reviews: 148


## Task 3: Multilingual Sentiment Translation (4 Marks)

**Steps:**
1. Sample 100 reviews (40 Pos, 40 Neg, 20 Neu), prioritize shorter reviews
2. Translate English $\rightarrow$ Hindi using `deep-translator` (`GoogleTranslator`)
3. Back-translate Hindi $\rightarrow$ English, compute BLEU score (threshold ≥ 0.3)
4. Check sentiment preservation on back-translated text
5. Manually verify 5 random samples
6. Save `bilingual_reviews.csv` with `bleu_score` and `quality_flag` columns

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from deep_translator import GoogleTranslator
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize

In [ ]:
# Load gold dataset (IMPORTANT: use only gold for training baseline)
gold_df = pd.read_csv("STT_A3/gold_standard_100.csv")

X = gold_df["review"]
y = gold_df["label"]

# TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=5000)
X_vec = vectorizer.fit_transform(X)

# Logistic Regression model
model = LogisticRegression(max_iter=200)
model.fit(X_vec, y)

print(" Model trained")

 Model trained


In [ ]:
#  3a. Strategic sampling

# Load your consolidated dataset
df = pd.read_csv("consolidated_base.csv")

# Add length column to prioritize shorter reviews
df["length"] = df["review"].apply(len)

# Sort by shortest first
df = df.sort_values("length")

# Sample required counts
pos = df[df["label"] == "Positive"].head(40)
neg = df[df["label"] == "Negative"].head(40)
neu = df[df["label"] == "Neutral"].head(20)

# Combine and shuffle
sample_df = pd.concat([pos, neg, neu]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Sampled dataset size:", len(sample_df))

Sampled dataset size: 100


In [ ]:
#  3b. Translation pipeline
# TODO: Translate each review English $\rightarrow$ Hindi using GoogleTranslator(source='en', target='hi')
# Add time.sleep(0.5) between calls to avoid rate limits

def translate_to_hindi(text):
    return GoogleTranslator(source='en', target='hi').translate(text)

# Apply translation
sample_df["hindi"] = None

for i in range(len(sample_df)):
    try:
        sample_df.loc[i, "hindi"] = translate_to_hindi(sample_df.loc[i, "review"])
        time.sleep(1.0)  # avoid rate limit
    except Exception as e:
        print(f"Error at index {i}:", e)
        sample_df.loc[i, "hindi"] = ""

Batch failed (0-9): Server Error: You made too many requests to the server.According to google, you are allowed to make 5 requests per secondand up to 200k requests per day. You can wait and try again later oryou can try the translate_batch function


KeyboardInterrupt: 

In [ ]:
#  3c. Back-translation & BLEU
# Translate Hindi $\rightarrow$ English
# Compute sentence BLEU between original and back-translated
# quality_flag = "PASS" if BLEU >= 0.3, else "FAIL"


def back_translate(text):
    return GoogleTranslator(source='hi', target='en').translate(text)

def compute_bleu(original, back_translated):
    smoothie = SmoothingFunction().method1
    ref = [word_tokenize(original.lower())]
    cand = word_tokenize(back_translated.lower())
    return sentence_bleu(ref, cand, smoothing_function=smoothie)

sample_df["back_translated"] = None
sample_df["bleu_score"] = 0.0
sample_df["quality_flag"] = "FAIL"

for i in range(len(sample_df)):
    try:
        back = back_translate(sample_df.loc[i, "hindi"])
        sample_df.loc[i, "back_translated"] = back

        bleu = compute_bleu(sample_df.loc[i, "review"], back)
        sample_df.loc[i, "bleu_score"] = bleu

        if bleu >= 0.3:
            sample_df.loc[i, "quality_flag"] = "PASS"

        time.sleep(0.5)

    except Exception as e:
        print(f"Error at index {i}:", e)
        sample_df.loc[i, "back_translated"] = ""

In [ ]:
#  3d. Sentiment preservation check
# TODO: Predict sentiment on back-translated text, compare with original label

def sentiment_match(text, label):
    pred = model.predict(vectorizer.transform([text]))[0]
    return pred == label

# Update quality flag with sentiment check
for i in range(len(sample_df)):
    back_text = sample_df.loc[i, "back_translated"]
    label = sample_df.loc[i, "label"]

    if sample_df.loc[i, "quality_flag"] == "PASS":
        if not sentiment_match(back_text, label):
            sample_df.loc[i, "quality_flag"] = "FAIL"

In [ ]:
#  3e. Manual verification
# TODO: Print 5 random samples for inspection

print("\n Manual inspection samples:\n")
print(sample_df.sample(5)[["review", "hindi", "back_translated", "label"]])


#  3f. Save
# TODO: Save bilingual_reviews.csv with columns:
# review, label, hindi, back_translated, bleu_score, quality_flag

sample_df.to_csv("bilingual_reviews.csv", index=False)

print("Saved as bilingual_reviews.csv")

## Task 4: Multimodal Audio Generation (4 Marks)

**Steps:**
1. Select 30 reviews (10 per class) of varying lengths
2. Use `gTTS` to generate audio (`tld="com"`), convert mp3$\rightarrow$wav via `librosa`+`soundfile`
3. Extract audio features with `librosa`: duration, spectral centroid, zero crossing rate, MFCCs
4. Use `openai-whisper` (tiny model) to transcribe audio back to text, compute WER
5. Save `audio_samples/` folder, `audio_features.csv`, `audio_validation.csv`

In [ ]:
!pip install gtts

In [ ]:
from gtts import gTTS
import librosa, soundfile as sf

#  4a. Select 30 reviews (10 per class)
# TODO: Sample from consolidated_base, mix short and long reviews
import pandas as pd

df = pd.read_csv("consolidated_base.csv")

# Add length column
df["length"] = df["review"].apply(len)

def sample_mixed(df_class, n=10):
    df_class = df_class.sort_values("length")
    half = n // 2
    short = df_class.head(half)
    long = df_class.tail(n - half)
    return pd.concat([short, long])

pos = sample_mixed(df[df["label"] == "Positive"], 10)
neg = sample_mixed(df[df["label"] == "Negative"], 10)
neu = sample_mixed(df[df["label"] == "Neutral"], 10)

audio_df = pd.concat([pos, neg, neu]).reset_index(drop=True)

print("Selected samples:", len(audio_df))

In [ ]:
#  4b. TTS generation
from gtts import gTTS
import librosa
import soundfile as sf
import os

os.makedirs("audio_samples", exist_ok=True)

file_paths = []

for i, row in audio_df.iterrows():
    text = row["review"]

    try:
        mp3_path = f"audio_samples/sample_{i}.mp3"
        wav_path = f"audio_samples/sample_{i}.wav"

        # Generate TTS
        tts = gTTS(text=text, lang='en', tld='com') # com used for US accent
        tts.save(mp3_path)

        # Convert mp3 → wav, .wav is better for analysis & ML models, that's why we are saving .wav as well.
        y, sr = librosa.load(mp3_path, sr=None)  # sr=22050 (default), hence sr = None to keep the original one, and avoid distortion in audio due to sampling.
        sf.write(wav_path, y, sr)

        file_paths.append(wav_path)

    except Exception as e:
        print(f"Error at {i}:", e)
        file_paths.append(None)

audio_df["wav_path"] = file_paths

# TODO: For each review, generate audio with gTTS (tld="com")
# Save as mp3, then load with librosa and re-save as .wav via soundfile


In [ ]:
#  4c. Audio feature extraction
# TODO: For each wav, extract with librosa:
#   - duration (librosa.get_duration)
#   - spectral centroid (librosa.feature.spectral_centroid)
#   - zero crossing rate (librosa.feature.zero_crossing_rate)
#   - MFCCs (librosa.feature.mfcc, n_mfcc=13, take mean)
# Save audio_features.csv

import numpy as np

features = []

for i, row in audio_df.iterrows():
    path = row["wav_path"]

    try:
        y, sr = librosa.load(path, sr=None)

        duration = librosa.get_duration(y=y, sr=sr)

        spectral_centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)) # Center of mass of frequency spectrum = spectral centroid
  # It indicates brightness of sound. High -> sharp/bright

        zcr = np.mean(librosa.feature.zero_crossing_rate(y)) # Number of times signal crosses zero axis. High ZCR = noisy/high freq. Low = smooth signal.

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13) # Mel-Frequency Cepstral Coefficients. Used in: Speech recognition and Audio classification
        mfcc_mean = np.mean(mfcc, axis=1)  # 13 values per audio, mfcc mimics human hearing perception.

        feature_row = {
            "file": path,
            "duration": duration,
            "spectral_centroid": spectral_centroid,
            "zcr": zcr
        }

        # Add MFCCs in
        for j in range(13):
            feature_row[f"mfcc_{j+1}"] = mfcc_mean[j] # adding mfcc_1 to mfcc_13 for each audio signal.

        features.append(feature_row)

    except Exception as e:
        print(f"Feature error at {i}:", e)

features_df = pd.DataFrame(features)
features_df.to_csv("audio_features.csv", index=False)

print("Saved audio_features.csv")



In [ ]:
!pip install openai-whisper

In [ ]:
#  4d. Whisper round-trip validation, whisper is speech to text library in OpenAI.
import whisper

_whisper_model = whisper.load_model("tiny") # tiny is the smallest model, it's fast

# TODO: Transcribe each wav with Whisper
# Compute WER (word-level Levenshtein distance / reference word count)
# Flag samples with WER > 0.25
# Save audio_validation.csv

# reference = original text, hypothesis = whisper output
def wer(reference, hypothesis):
    ref_words = reference.lower().split()
    hyp_words = hypothesis.lower().split()

    # Levenshtein distance calculation using Dynamic Programming
    dp = [[0]*(len(hyp_words)+1) for _ in range(len(ref_words)+1)] # created DP Matrix

    for i in range(len(ref_words)+1):
        dp[i][0] = i
    for j in range(len(hyp_words)+1):
        dp[0][j] = j

    for i in range(1, len(ref_words)+1):
        for j in range(1, len(hyp_words)+1):
            if ref_words[i-1] == hyp_words[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],    # deletion
                    dp[i][j-1],    # insertion
                    dp[i-1][j-1]   # substitution
                )

    return dp[-1][-1] / len(ref_words) if len(ref_words) > 0 else 0

validation_results = []

for i, row in audio_df.iterrows():
    path = row["wav_path"]
    original = row["review"]

    try:
        result = _whisper_model.transcribe(path)
        transcript = result["text"]

        error = wer(original, transcript)
        flag = "FAIL" if error > 0.25 else "PASS"

        validation_results.append({
            "file": path,
            "original": original,
            "transcript": transcript,
            "wer": error,
            "quality_flag": flag
        })

    except Exception as e:
        print(f"Whisper error at {i}:", e)

validation_df = pd.DataFrame(validation_results)

validation_df.to_csv("audio_validation.csv", index=False)
print("Saved audio_validation.csv")

In [ ]:
display(validation_df)

## Task 5: Final Dataset Assembly & Model Evaluation (2 Marks)

**Steps:**
1. Merge all datasets: `consolidated_base.csv` + `augmented_classical.csv` + `llm_generated_300.csv` (excluding flagged) + English text from `bilingual_reviews.csv`
2. Deduplicate $\rightarrow$ save `final_augmented_dataset.csv`
3. Use `BlackBoxEvaluator` from `evaluator.py` to compare baseline vs augmented accuracy

In [ ]:
from STT_A3.evaluator import BlackBoxEvaluator

# 5a. Assemble final dataset

# Load datasets
base_df = pd.read_csv("consolidated_base.csv")
aug_classical_df = pd.read_csv("augmented_classical.csv")
llm_df = pd.read_csv("llm_generated_300.csv")
bilingual_df = pd.read_csv("bilingual_reviews.csv")

# Extract English text from bilingual dataset

# Adjust column name if needed (e.g., "english_review")
display(bilingual_df)
if "english_review" in bilingual_df.columns:
    bilingual_df = bilingual_df.rename(columns={"english_review": "review"})
elif "review_en" in bilingual_df.columns:
    bilingual_df = bilingual_df.rename(columns={"review_en": "review"})

# Keep only necessary columns (review + label)
bilingual_df = bilingual_df[["review", "label"]]

# Merge all datasets
final_df = pd.concat([
    base_df,
    aug_classical_df,
    llm_df,
    bilingual_df
], ignore_index=True)

# Deduplicate
final_df = final_df.drop_duplicates(subset=["review"])

# Save final dataset
final_df.to_csv("final_augmented_dataset.csv", index=False)

print(f"Final dataset size: {len(final_df)}")
print("Saved as final_augmented_dataset.csv")



# 5b. Black-Box Evaluation

evaluator = BlackBoxEvaluator()

# Load test set
test_df = pd.read_csv("gold_standard_100.csv")

# Baseline evaluation
baseline_acc = evaluator.run_evaluation(base_df, test_df)

# Augmented evaluation
augmented_acc = evaluator.run_evaluation(final_df, test_df)

# Print comparison
print(f"\nBaseline accuracy:   {baseline_acc:.2%}")
print(f"Augmented accuracy: {augmented_acc:.2%}")
print(f"Improvement:        {augmented_acc - baseline_acc:+.2%}")